In [3]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import lightgbm

In [4]:
# Reading the Dataset
dataset = pd.read_csv('insurance_pre.csv')
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [5]:
dataset.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges'], dtype='object')

In [6]:
dataset=pd.get_dummies(dataset, dtype=int, drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


dataset

In [7]:
indep=dataset[['age', 'sex_male', 'bmi', 'children', 'smoker_yes']]
dep=dataset[['charges']]

In [8]:
indep

,age,sex_male,bmi,children,smoker_yes
0,19,0,27.900,0,1
1,18,1,33.770,1,0
2,28,1,33.000,3,0
3,33,1,22.705,0,0
4,32,1,28.880,0,0
...,...,...,...,...,...
1333,50,1,30.970,3,0
1334,18,0,31.920,0,0
1335,18,0,36.850,0,0
1336,21,0,25.800,0,0


In [9]:
dep

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [10]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)


In [11]:
X_train

,age,sex_male,bmi,children,smoker_yes
482,18,0,31.35,0,0
338,50,1,32.30,1,1
356,46,1,43.89,3,0
869,25,0,24.30,3,0
182,22,1,19.95,3,0
...,...,...,...,...,...
763,27,1,26.03,0,0
835,42,1,35.97,2,0
1216,40,1,25.08,0,0
559,19,1,35.53,0,0


In [12]:
X_train.shape

(892, 5)

In [13]:
y_train

,charges
482,1622.1885
338,41919.0970
356,8944.1151
869,4391.6520
182,4005.4225
...,...
763,3070.8087
835,7160.3303
1216,5415.6612
559,1646.4297


In [14]:
y_train.shape

(892, 1)

In [15]:
X_test

,age,sex_male,bmi,children,smoker_yes
578,52,1,30.200,1,0
610,47,0,29.370,1,0
569,48,1,40.565,2,1
1034,61,1,38.380,0,0
198,51,0,18.050,0,0
...,...,...,...,...,...
261,20,0,26.840,1,1
1271,25,0,34.485,0,0
1313,19,0,34.700,2,1
2,28,1,33.000,3,0


In [16]:
X_test.shape

(446, 5)

In [17]:
y_test

,charges
578,9724.53000
610,8547.69130
569,45702.02235
1034,12950.07120
198,9644.25250
...,...
261,17085.26760
1271,3021.80915
1313,36397.57600
2,4449.46200


In [18]:
y_test.shape

(446, 1)

In [26]:
#Traditional Gradient Boosting Decision Trees - gbdt
from lightgbm import LGBMRegressor
regressor = LGBMRegressor(boosting_type="gbdt",     num_leaves=31,     max_depth=-1,     learning_rate=0.1,     n_estimators=100)
regressor.fit(X_train, y_train)
y_pred=regressor.predict(X_test)
from sklearn.metrics import r2_score
r_score=r2_score(y_test,y_pred)
print(r_score)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000192 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 312
[LightGBM] [Info] Number of data points in the train set: 892, number of used features: 5
[LightGBM] [Info] Start training from score 13138.323530
0.866506069829381


In [25]:
#Dropout-based boosting - dart
from lightgbm import LGBMRegressor
regressor = LGBMRegressor(boosting_type="dart",     num_leaves=31,     max_depth=-1,     learning_rate=0.1,     n_estimators=100)
regressor.fit(X_train, y_train)
y_pred=regressor.predict(X_test)
from sklearn.metrics import r2_score
r_score=r2_score(y_test,y_pred)
print(r_score)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000030 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 312
[LightGBM] [Info] Number of data points in the train set: 892, number of used features: 5
[LightGBM] [Info] Start training from score 13138.323530
0.8696252230715567


In [24]:
#Random Forest mode - rf
#Mandatory parameters subsample=0.8,     subsample_freq=1,     colsample_bytree=0.8
from lightgbm import LGBMRegressor
regressor = LGBMRegressor(boosting_type="rf",  n_estimators=100, subsample=0.8,     subsample_freq=1,     colsample_bytree=0.8)
regressor.fit(X_train, y_train)
y_pred=regressor.predict(X_test)
from sklearn.metrics import r2_score
r_score=r2_score(y_test,y_pred)
print(r_score)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000223 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 312
[LightGBM] [Info] Number of data points in the train set: 892, number of used features: 5
[LightGBM] [Info] Start training from score 13138.323530
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,